# Analyzing PerfGazer Reports

This notebook shows how to analyze PerfGazer reports to identify performance bottlenecks.

**Prerequisites:** Run `01-quickstart.ipynb` first to generate sample reports.

## 1. Setup

In [ ]:
// Local setup only
import $ivy.`org.apache.spark::spark-sql:3.5.2`

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("PerfGazer Analysis")
  .master("local[*]")  // Remove on Databricks
  .getOrCreate()

import spark.implicits._

// Path to PerfGazer reports
val reportsDir = "/tmp/perfgazer-demo"

## 2. Load Reports

In [ ]:
val jobs = spark.read.json(s"$reportsDir/job-reports-*.json")
val stages = spark.read.json(s"$reportsDir/stage-reports-*.json")
val sqls = spark.read.json(s"$reportsDir/sql-reports-*.json")

jobs.createOrReplaceTempView("job")
stages.createOrReplaceTempView("stage")
sqls.createOrReplaceTempView("sql")

println(s"Loaded ${jobs.count()} jobs, ${stages.count()} stages, ${sqls.count()} SQL queries")

## 3. Job Duration Analysis

In [ ]:
// Job durations ranked by time
spark.sql("""
  SELECT 
    jobId,
    jobName,
    (jobEndTime - jobStartTime) as durationMs,
    size(stages) as numStages
  FROM job
  ORDER BY durationMs DESC
""").show(20, truncate = false)

## 4. Stage Metrics Analysis

In [ ]:
// Show stage schema to see available metrics
stages.printSchema()

In [ ]:
// Stage metrics summary
spark.sql("""
  SELECT 
    stageId,
    attempt,
    execRunNs / 1000000 as execRunMs,
    execCpuNs / 1000000 as execCpuMs,
    execJvmGcNs / 1000000 as gcMs,
    shuffleReadBytes,
    shuffleWriteBytes,
    memoryBytesSpilled,
    diskBytesSpilled
  FROM stage
  ORDER BY execRunNs DESC
""").show(truncate = false)

## 5. Identify Spill Issues

Spilling to disk indicates memory pressure and can significantly slow down jobs.

In [ ]:
// Find stages with spill
spark.sql("""
  SELECT 
    stageId,
    attempt,
    memoryBytesSpilled,
    diskBytesSpilled,
    ROUND(diskBytesSpilled / 1024 / 1024, 2) as diskSpillMB
  FROM stage
  WHERE memoryBytesSpilled > 0 OR diskBytesSpilled > 0
  ORDER BY diskBytesSpilled DESC
""").show(truncate = false)

## 6. SQL Query Analysis

In [ ]:
// SQL queries with their node counts (complexity indicator)
spark.sql("""
  SELECT 
    sqlId,
    description,
    size(nodes) as numNodes
  FROM sql
  ORDER BY numNodes DESC
""").show(truncate = false)

In [ ]:
// Explode SQL nodes to see execution plan details
spark.sql("""
  SELECT 
    sqlId,
    node.nodeName,
    node.coordinates,
    node.isLeaf
  FROM sql
  LATERAL VIEW explode(nodes) AS node
  ORDER BY sqlId, node.coordinates
""").show(50, truncate = false)

## 7. Shuffle Analysis

Large shuffles can be a major performance bottleneck.

In [ ]:
// Shuffle statistics
spark.sql("""
  SELECT 
    stageId,
    attempt,
    ROUND(shuffleReadBytes / 1024 / 1024, 2) as shuffleReadMB,
    ROUND(shuffleWriteBytes / 1024 / 1024, 2) as shuffleWriteMB,
    ROUND(readBytes / 1024 / 1024, 2) as inputReadMB,
    ROUND(writeBytes / 1024 / 1024, 2) as outputWriteMB
  FROM stage
  WHERE shuffleReadBytes > 0 OR shuffleWriteBytes > 0
  ORDER BY shuffleReadBytes + shuffleWriteBytes DESC
""").show(truncate = false)

## 8. Cleanup

In [ ]:
spark.stop()